# 汇总所有实验的 Emean 结果

把各器官 `04_other/patient_level_exports/<split>/predicted_emean_image_level.csv`（由导出脚本产生）
汇总成两张表：

1. **多器官汇总表**：肾脏 / 胰腺 / 腮腺 / 颌下腺的 train / validation / internal_test / external_test。
2. **肝脏表（单独）**：肝脏补充实验单独汇总。

只读已有产物，不重新训练；缺少产物时跳过并在输出里说明。


In [ ]:
import sys
from pathlib import Path

import pandas as pd

REPO_ROOT = next(
    path
    for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "common" / "paths.py").is_file()
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from common.metrics import summarize_regression
from common.paths import EXPERIMENT

SUMMARY_DIR = EXPERIMENT / "summary"
SUMMARY_DIR.mkdir(parents=True, exist_ok=True)
GENERALISATION_ORGANS = ["kidney", "pancreas", "parotid", "submandibular"]
SPLIT_ORDER = ["train", "validation", "internal_test", "external_test"]

In [ ]:
def summarise_export(image_csv):
    frame = pd.read_csv(image_csv)
    frame = frame.rename(columns={"true_emean": "emean_true", "predicted_emean": "emean_pred"})
    patient_frame = (
        frame.groupby("patient_id", as_index=False)[["emean_true", "emean_pred"]]
        .mean()
        .reset_index(drop=True)
    )
    return summarize_regression(frame, patient_frame), len(frame), len(patient_frame)


rows = []
missing = []
for organ in GENERALISATION_ORGANS:
    export_root = EXPERIMENT / organ / "04_other" / "patient_level_exports"
    if not export_root.is_dir():
        missing.append(str(export_root))
        continue
    for split_dir in sorted(export_root.iterdir()):
        image_csv = split_dir / "predicted_emean_image_level.csv"
        if not image_csv.is_file():
            continue
        metrics, images, patients = summarise_export(image_csv)
        for level in ("image_level", "patient_level"):
            for metric, value in metrics[level].items():
                rows.append(
                    {
                        "organ": organ,
                        "split": split_dir.name,
                        "level": level,
                        "metric": metric,
                        "value": value,
                        "images": images,
                        "patients": patients,
                    }
                )

generalisation = pd.DataFrame(rows)
if not generalisation.empty:
    generalisation["split"] = pd.Categorical(
        generalisation["split"], categories=SPLIT_ORDER, ordered=True
    )
    generalisation = generalisation.sort_values(["organ", "split", "level", "metric"])
    generalisation.to_csv(SUMMARY_DIR / "metrics_summary.csv", index=False)
print("多器官汇总表:", "已写出" if not generalisation.empty else "无产物")
if missing:
    print("缺少导出目录（先跑各器官 04_other 的导出脚本）:")
    for path in missing:
        print("  ", path)
generalisation.head(12)

In [ ]:
# 肝脏：补充实验，单独一张表
liver_outputs = EXPERIMENT / "liver" / "02_ablation" / "outputs"
liver_image_csv = liver_outputs / "final_pretrained_emean_predictions.csv"
liver_rows = []
if liver_image_csv.is_file():
    liver_frame = pd.read_csv(liver_image_csv)
    liver_patient = (
        liver_frame.groupby("patient_id", as_index=False)[["emean_true", "emean_pred"]]
        .mean()
        .reset_index(drop=True)
    )
    liver_metrics = summarize_regression(liver_frame, liver_patient)
    for level in ("image_level", "patient_level"):
        for metric, value in liver_metrics[level].items():
            liver_rows.append(
                {
                    "organ": "liver",
                    "data_mode": "supplementary",
                    "level": level,
                    "metric": metric,
                    "value": value,
                    "images": len(liver_frame),
                    "patients": len(liver_patient),
                }
            )
    liver_table = pd.DataFrame(liver_rows)
    liver_table.to_csv(SUMMARY_DIR / "metrics_summary_liver.csv", index=False)
    print("肝脏表已写出（单独汇总）")
else:
    liver_table = pd.DataFrame()
    print("没有肝脏产物，跳过：", liver_image_csv)
liver_table